# SolarShield — Notebook 04: Validation (MANDATORY before handoff)
This notebook **must pass all assertions** before handing `.pkl` files to the backend member.  
It loads both models cold, runs 10 hand-crafted samples, and asserts all output constraints.

> ✅ If this notebook completes without errors → safe to hand off  
> ❌ If any assertion fails → retrain before handing off

In [9]:
import pandas as pd
import numpy as np
import joblib
import json

# ── Load feature order from the canonical JSON (same as backend reads) ────────
with open('../ml_models/features.json') as f:
    FEATURE_ORDER = json.load(f)['feature_order']

assert len(FEATURE_ORDER) == 12, f'Expected 12 features, got {len(FEATURE_ORDER)}'
print('feature_order loaded from features.json:', FEATURE_ORDER)

# ── Load models cold (simulates backend startup) ──────────────────────────────
rf_model  = joblib.load('../ml_models/rf_classifier.pkl')
xgb_model = joblib.load('../ml_models/xgb_regressor.pkl')
print('Both models loaded successfully.')

feature_order loaded from features.json: ['voltage_mean', 'voltage_std', 'current_mean', 'current_std', 'power_mean', 'power_std', 'lux_mean', 'temperature_mean', 'humidity_mean', 'power_rate_of_change', 'voltage_rate_of_change', 'efficiency_ratio']
Both models loaded successfully.


## Hand-crafted test samples
10 rows designed to cover Normal, Degraded, and Fault operating conditions.

In [10]:
# fmt: off
samples = [
    # Normal operating conditions (high lux, good power)
    {'voltage_mean':24.5,'voltage_std':0.2,'current_mean':3.8,'current_std':0.1,
     'power_mean':93.0,'power_std':2.0,'lux_mean':85000,'temperature_mean':28,
     'humidity_mean':55,'power_rate_of_change':0.1,'voltage_rate_of_change':0.0,
     'efficiency_ratio':93.0/(85000/1000+0.001)},

    # Normal — slightly lower irradiance
    {'voltage_mean':24.2,'voltage_std':0.3,'current_mean':3.5,'current_std':0.15,
     'power_mean':84.0,'power_std':3.0,'lux_mean':75000,'temperature_mean':30,
     'humidity_mean':60,'power_rate_of_change':0.05,'voltage_rate_of_change':0.0,
     'efficiency_ratio':84.0/(75000/1000+0.001)},

    # Degraded — reduced efficiency
    {'voltage_mean':22.0,'voltage_std':0.8,'current_mean':2.5,'current_std':0.3,
     'power_mean':55.0,'power_std':5.0,'lux_mean':70000,'temperature_mean':35,
     'humidity_mean':70,'power_rate_of_change':-0.2,'voltage_rate_of_change':-0.05,
     'efficiency_ratio':55.0/(70000/1000+0.001)},

    # Degraded — partial shading
    {'voltage_mean':21.5,'voltage_std':1.2,'current_mean':2.0,'current_std':0.5,
     'power_mean':43.0,'power_std':8.0,'lux_mean':60000,'temperature_mean':33,
     'humidity_mean':65,'power_rate_of_change':-0.5,'voltage_rate_of_change':-0.1,
     'efficiency_ratio':43.0/(60000/1000+0.001)},

    # Fault — very low efficiency ratio
    {'voltage_mean':18.0,'voltage_std':3.0,'current_mean':0.8,'current_std':0.8,
     'power_mean':14.0,'power_std':10.0,'lux_mean':80000,'temperature_mean':40,
     'humidity_mean':80,'power_rate_of_change':-1.5,'voltage_rate_of_change':-0.5,
     'efficiency_ratio':14.0/(80000/1000+0.001)},

    # Fault — night / zero output (numerical edge case)
    {'voltage_mean':0.0,'voltage_std':0.0,'current_mean':0.0,'current_std':0.0,
     'power_mean':0.0,'power_std':0.0,'lux_mean':10,'temperature_mean':22,
     'humidity_mean':75,'power_rate_of_change':0.0,'voltage_rate_of_change':0.0,
     'efficiency_ratio':0.0/(10/1000+0.001)},

    # Normal — morning ramp-up
    {'voltage_mean':23.8,'voltage_std':0.5,'current_mean':3.0,'current_std':0.2,
     'power_mean':71.0,'power_std':4.0,'lux_mean':60000,'temperature_mean':25,
     'humidity_mean':72,'power_rate_of_change':1.2,'voltage_rate_of_change':0.1,
     'efficiency_ratio':71.0/(60000/1000+0.001)},

    # Degraded — hot day, thermal derating
    {'voltage_mean':22.5,'voltage_std':0.6,'current_mean':2.8,'current_std':0.2,
     'power_mean':63.0,'power_std':3.5,'lux_mean':90000,'temperature_mean':48,
     'humidity_mean':30,'power_rate_of_change':-0.1,'voltage_rate_of_change':-0.02,
     'efficiency_ratio':63.0/(90000/1000+0.001)},

    # Fault — panel disconnected
    {'voltage_mean':5.0,'voltage_std':4.0,'current_mean':0.1,'current_std':0.1,
     'power_mean':0.5,'power_std':0.5,'lux_mean':85000,'temperature_mean':30,
     'humidity_mean':60,'power_rate_of_change':-2.0,'voltage_rate_of_change':-1.0,
     'efficiency_ratio':0.5/(85000/1000+0.001)},

    # Normal — overcast but functional
    {'voltage_mean':23.5,'voltage_std':0.4,'current_mean':1.5,'current_std':0.1,
     'power_mean':35.0,'power_std':2.0,'lux_mean':30000,'temperature_mean':27,
     'humidity_mean':85,'power_rate_of_change':0.0,'voltage_rate_of_change':0.0,
     'efficiency_ratio':35.0/(30000/1000+0.001)},
]
# fmt: on

X_val = pd.DataFrame(samples)[FEATURE_ORDER]  # enforce exact feature order
print(f'Validation input shape: {X_val.shape}')
X_val.head(3)

Validation input shape: (10, 12)


,voltage_mean,voltage_std,current_mean,current_std,power_mean,power_std,lux_mean,temperature_mean,humidity_mean,power_rate_of_change,voltage_rate_of_change,efficiency_ratio
0,24.5,0.2,3.8,0.10,93.0,2.0,85000,28,55,0.10,0.00,1.094105
1,24.2,0.3,3.5,0.15,84.0,3.0,75000,30,60,0.05,0.00,1.119985
2,22.0,0.8,2.5,0.30,55.0,5.0,70000,35,70,-0.20,-0.05,0.785703


## Run predictions and assert constraints

In [11]:
FAULT_LABELS = {0: 'Normal', 1: 'Degraded', 2: 'Fault'}
results = []

for i, row in X_val.iterrows():
    x_row = pd.DataFrame([row])

    # Classifier
    fault_class = int(rf_model.predict(x_row)[0])
    fault_label = FAULT_LABELS[fault_class]

    # Regressor
    xgb_out    = xgb_model.predict(x_row)[0]
    efficiency = float(np.clip(xgb_out[0], 0, 100))
    maint_days = int(np.clip(round(xgb_out[1]), 0, 365))

    # ── Assertions ────────────────────────────────────────────────────────────
    assert fault_class in (0, 1, 2),           f'Row {i}: fault_class {fault_class} not in {{0,1,2}}'
    assert 0.0 <= efficiency <= 100.0,         f'Row {i}: efficiency_score {efficiency} out of [0,100]'
    assert 0   <= maint_days <= 365,           f'Row {i}: maintenance_days {maint_days} out of [0,365]'

    results.append({
        'sample': i,
        'fault_class': fault_class,
        'fault_label': fault_label,
        'efficiency_score': round(efficiency, 2),
        'maintenance_days': maint_days
    })

results_df = pd.DataFrame(results)
print('=== All 10 Validation Predictions ===')
print(results_df.to_string(index=False))

=== All 10 Validation Predictions ===
 sample  fault_class fault_label  efficiency_score  maintenance_days
      0            0      Normal             89.97                80
      1            0      Normal             83.11                81
      2            0      Normal             57.57                81
      3            0      Normal             40.35                83
      4            0      Normal             10.72                85
      5            2       Fault              0.42                83
      6            0      Normal             78.80                81
      7            0      Normal             65.84                81
      8            0      Normal              9.37                85
      9            0      Normal             36.65                82


In [12]:
# ── Final checklist ───────────────────────────────────────────────────────────
print('\n=== Pre-Handoff Checklist ===')

import os
checks = [
   ('ml_models/rf_classifier.pkl exists',  os.path.isfile('../ml_models/rf_classifier.pkl')),
   ('ml_models/xgb_regressor.pkl exists',  os.path.isfile('../ml_models/xgb_regressor.pkl')),
   ('ml_models/features.json exists',      os.path.isfile('../ml_models/features.json')),
    ('features.json has 12 features',        len(FEATURE_ORDER) == 12),
    ('All fault_class in {0,1,2}',           results_df['fault_class'].isin([0,1,2]).all()),
    ('All efficiency in [0,100]',            results_df['efficiency_score'].between(0,100).all()),
    ('All maintenance_days in [0,365]',      results_df['maintenance_days'].between(0,365).all()),
]

all_pass = True
for desc, passed in checks:
    status = '✅ PASS' if passed else '❌ FAIL'
    print(f'  {status}  {desc}')
    all_pass = all_pass and passed

print()
if all_pass:
    print('🎉 ALL CHECKS PASSED — models are ready for backend handoff.')
else:
    raise RuntimeError('One or more checks failed. Fix issues before handing off .pkl files.')


=== Pre-Handoff Checklist ===
  ✅ PASS  ml_models/rf_classifier.pkl exists
  ✅ PASS  ml_models/xgb_regressor.pkl exists
  ✅ PASS  ml_models/features.json exists
  ✅ PASS  features.json has 12 features
  ✅ PASS  All fault_class in {0,1,2}
  ✅ PASS  All efficiency in [0,100]
  ✅ PASS  All maintenance_days in [0,365]

🎉 ALL CHECKS PASSED — models are ready for backend handoff.
